In [ ]:
# Parámetros
plant_codes  = None
period_start  = None     # Si vacío y incremental=True, se calcula por planta como max(Date) en scada_raw
period_end    = None

# Modo incremental: para cada planta, lee del gold solo desde max(Date) ya presente en scada_raw.
# Si scada_raw aún no tiene datos para esa planta, la primera vez hace ingesta completa.
# Pon a False para forzar re-ingesta completa (period_start sigue respetándose como override).
incremental   = True
overlap_hours = 0        # Margen de seguridad: re-lee las últimas N horas por si llegaron datos tardíos.

In [2]:
def _norm_plant_codes(v):
    """Normaliza plant_codes a lista (una o varias). None/'' = todas.
    Acepta lista/tupla, JSON array o cadena separada por comas."""
    if v is None:
        return []
    if isinstance(v, (list, tuple)):
        return [str(x).strip() for x in v if str(x).strip()]
    s = str(v).strip()
    if not s:
        return []
    try:
        import json as _json
        parsed = _json.loads(s)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except (ValueError, TypeError):
        pass
    return [p.strip() for p in s.split(',') if p.strip()]


import io
import pandas as pd
from datetime import datetime
from pathlib import Path
from delta.tables import DeltaTable
from pyspark.sql import Row
from pyspark.sql.functions import col, lit, current_timestamp

LAKEHOUSE_FILES = '/lakehouse/default/Files'
CONFIG_XLSX     = f'{LAKEHOUSE_FILES}/Raw PVSyst1/CONFIG/plants_config.xlsx'


ABFS_BASE = "abfss://0c4c3bbe-a6ae-404f-b1b7-7052bacfea11@onelake.dfs.fabric.microsoft.com/ae7e1482-8f15-4b1b-8954-8ee500382179"


def _read(sheet, table=None):
    """Lee una TABLA ESTRUCTURADA del Excel de config: 1) por NOMBRE de tabla
    (case-insensitive, en cualquier hoja); 2) si no, la (primera) tabla de la hoja
    indicada; 3) ultimo recurso, la hoja completa. Robusto ante renombrados."""
    local = '/tmp/plants_config.xlsx'
    if not Path(local).exists():
        abfs_path = ABFS_BASE + '/Files/Raw PVSyst1/CONFIG/plants_config.xlsx'
        row = spark.read.format('binaryFile').load(abfs_path).first()
        with open(local, 'wb') as _f:
            _f.write(row['content'])
    import openpyxl
    wb = openpyxl.load_workbook(local, data_only=True)
    def _df(ws, ref):
        _d = [[c.value for c in _r] for _r in ws[ref]]
        return pd.DataFrame(_d[1:], columns=_d[0]) if _d else pd.DataFrame()
    if table:
        for ws in wb.worksheets:
            _tm = {k.lower(): k for k in ws.tables}
            if table.lower() in _tm:
                return _df(ws, ws.tables[_tm[table.lower()]].ref)
    if sheet in wb.sheetnames:
        ws = wb[sheet]
        if ws.tables:
            return _df(ws, ws.tables[next(iter(ws.tables))].ref)
    return pd.read_excel(local, sheet_name=sheet)
def get_active_plants():
    df = _read('01_Plants', 'T01_plants')
    df = df[df['pr_execution'].astype(str).str.upper().isin(['TRUE','VERDADERO','1','SÍ','SI'])]
    _pl = _norm_plant_codes(plant_codes)
    if _pl:
        df = df[df['plant_code'].isin(_pl)]
    return df.to_dict(orient='records')


def get_scada_mapping(plant_code):
    m = _read('02_Sensors_Mapping', 'T02_sensor_mapping')
    # COMMON es la BASE (sensores estandar comunes); las filas de la planta
    # AÑADEN o SOBRESCRIBEN (no reemplazan). Antes era excluyente: si la planta
    # tenia 1 fila, se ignoraba COMMON y se perdian todos los demas sensores.
    mapping = {}
    for pc in ('COMMON', plant_code):
        for _, r in m[m['plant_code'] == pc].iterrows():
            orig = str(r['columna_original']).strip() if pd.notna(r['columna_original']) else None
            dest = str(r['columna_destino']).strip()  if pd.notna(r['columna_destino'])  else None
            if orig and dest:
                mapping[orig] = dest
    return mapping


def get_all_dest_columns():
    """Universo COMPLETO de columnas destino del mapping (todas las plantas).
    Garantiza que scada_raw tenga SIEMPRE el mismo esquema de sensores,
    independientemente de que una planta concreta traiga o no cada sensor."""
    m = _read('02_Sensors_Mapping', 'T02_sensor_mapping')
    dests = []
    for _, r in m.iterrows():
        d = str(r['columna_destino']).strip() if pd.notna(r['columna_destino']) else None
        if d and d not in dests:
            dests.append(d)
    if 'Date' not in dests:
        dests = ['Date'] + dests
    return dests

In [3]:
def read_and_normalize_scada(table_name, mapping, period_start=None, period_end=None):
    sdf = spark.table(table_name)

    # Toda columna de sensor debe estar en 02_Sensors_Mapping. Las del source que
    # NO esten mapeadas se consideran "no identificadas": se avisa y NO se procesan
    # (no entran en scada_raw). Date siempre se conserva.
    _ident   = set(mapping.keys()) | set(mapping.values()) | {'Date', 'date'}
    _unident = [c for c in sdf.columns if c not in _ident]
    if _unident:
        print(f"  AVISO: {len(_unident)} columna(s) del source NO identificada(s) en "
              f"02_Sensors_Mapping -> se IGNORAN (no entran en scada_raw): {_unident}")
    # Sensores mapeados que esta planta no trae en su tabla gold (quedaran null en scada_raw)
    _missing = [orig for orig in mapping.keys() if orig not in sdf.columns]
    if _missing:
        print(f"  info: sensores mapeados ausentes en el source (quedaran null): {_missing}")

    for orig, dest in mapping.items():
        if orig in sdf.columns and orig != dest:
            sdf = sdf.withColumnRenamed(orig, dest)

    if 'Date' not in sdf.columns:
        raise ValueError(f"La tabla {table_name} no contiene columna Date tras el mapeo")

    if period_start:
        sdf = sdf.filter(col('Date') >= lit(period_start).cast('timestamp'))
    if period_end:
        sdf = sdf.filter(col('Date') <= lit(period_end).cast('timestamp'))

    # Garantizar el esquema COMPLETO del mapping: toda columna destino definida en
    # 02_Sensors_Mapping debe existir en scada_raw aunque esta planta no la traiga
    # (queda null). Asi la tabla tiene tantas columnas de sensores como el mapping
    # y no depende del orden en que se procesan las plantas.
    all_dests = get_all_dest_columns()
    for d in all_dests:
        if d != 'Date' and d not in sdf.columns:
            sdf = sdf.withColumn(d, lit(None).cast('double'))
    keep = ['Date'] + [c for c in all_dests if c != 'Date']
    sdf = sdf.select(*[c for c in keep if c in sdf.columns])
    return sdf

In [ ]:
from datetime import timedelta


def get_last_ingested_date(plant_code):
    """Devuelve el max(Date) de scada_raw para esta planta, o None si aún no hay nada."""
    if not spark.catalog.tableExists('scada_raw'):
        return None
    row = spark.sql(f"""
        SELECT MAX(Date) AS mx
        FROM scada_raw
        WHERE plant_code = '{plant_code}'
    """).first()
    return row['mx']  # None si la planta nunca se ha ingerido


def merge_into_silver(sdf, plant_code, source_table):
    n = sdf.count()  # capturamos antes de añadir columnas de control
    sdf = (
        sdf
        .withColumn('plant_code',  lit(plant_code))
        .withColumn('source_file', lit(source_table))
        .withColumn('ingested_at', current_timestamp())
    )
    table = 'scada_raw'
    if spark.catalog.tableExists(table):
        spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
        target = DeltaTable.forName(spark, table)
        cond = "t.plant_code = s.plant_code AND t.Date = s.Date"
        # Cada planta tiene un set de sensores distinto (FRA no trae WS_01, etc.).
        # Mapear SOLO las columnas presentes en el source; UpdateAll/InsertAll
        # referenciaria columnas del destino ausentes en el source y fallaria.
        _set = {col: f"s.`{col}`" for col in sdf.columns}
        (target.alias('t')
            .merge(sdf.alias('s'), cond)
            .whenMatchedUpdate(set=_set)
            .whenNotMatchedInsert(values=_set)
            .execute())
    else:
        (sdf.write
            .format('delta')
            .option('mergeSchema', 'true')
            .partitionBy('plant_code')
            .mode('append')
            .saveAsTable(table))
    return n



def ensure_full_schema(table):
    """Garantiza que la tabla tenga TODAS las columnas destino del mapping
    (ausentes -> DOUBLE null). scada_raw tendra tantas columnas de sensores
    como 02_Sensors_Mapping."""
    if not spark.catalog.tableExists(table):
        return
    existing = set(spark.table(table).columns)
    for d in get_all_dest_columns():
        if d != 'Date' and d not in existing:
            spark.sql(f"ALTER TABLE {table} ADD COLUMNS (`{d}` DOUBLE)")
            print(f"  schema: +{d} en {table}")


ensure_full_schema('scada_raw')

results = []
for p in get_active_plants():
    plant   = p['plant_code']
    table   = 'gold.' + str(p['scada_table_name']).strip()
    print(f"\n→ {plant}: leyendo tabla {table}")
    try:
        # 1) Determinar period_start efectivo para esta planta
        eff_start = period_start  # override global del parámetro tiene prioridad absoluta
        if eff_start is None and incremental:
            last_date = get_last_ingested_date(plant)
            if last_date is not None:
                eff_start = (last_date - timedelta(hours=overlap_hours)).isoformat()
                print(f"  incremental: leyendo desde {eff_start}  (max actual {last_date}, overlap {overlap_hours}h)")
            else:
                print(f"  primera ingesta de {plant}: lectura completa")
        elif eff_start is None:
            print(f"  incremental=False: lectura completa")

        # 2) Leer + normalizar + mergear
        mapping = get_scada_mapping(plant)
        sdf     = read_and_normalize_scada(table, mapping, eff_start, period_end)
        n       = merge_into_silver(sdf, plant, table)
        status, error = 'ok', None
        print(f"  {n:,} filas merged en scada_raw")
    except Exception as e:
        n = 0; status = 'failed'; error = str(e)[:500]
        print(f"  FAILED: {error}")
    results.append(Row(
        plant_code=plant, file_path=f'onelake://{table}',
        file_kind='scada', file_size=None, row_count=n,
        period_start=None, period_end=None, resolution=None,
        status=status, error=error, ingested_at=datetime.utcnow(),
    ))

if results:
    rows_pd = pd.DataFrame([r.asDict() for r in results])
    for tcol in ('period_start', 'period_end', 'ingested_at'):
        rows_pd[tcol] = pd.to_datetime(rows_pd[tcol], errors='coerce')
    spark.createDataFrame(rows_pd).write.mode('append').format('delta').saveAsTable('ingested_files')
    print(f"\nLogged {len(results)} filas en ingested_files")

In [5]:
import json, notebookutils
summary = {
    'plants_processed': len(results),
    'ok':     sum(1 for r in results if r['status'] == 'ok'),
    'failed': sum(1 for r in results if r['status'] == 'failed'),
}
print(json.dumps(summary, indent=2))
notebookutils.notebook.exit(json.dumps(summary))
